In [ ]:
from __future__ import annotations

import os
import pathlib
from dataclasses import dataclass
import concurrent.futures
from typing import Dict, List, Optional, Set, Tuple

import duckdb

import sys
sys.path.insert(0, "../../../shared")
from locations import Location


# --- Config ---
@dataclass
class Config:
    duckdb_threads: int = min(8, os.cpu_count() or 1)
    files_per_chunk: int = 1_000  # how many parquet files per COUNT(*) round

CONFIG = Config()


def _sql_quote_path(p: str) -> str:
    """Quote a path for DuckDB SQL: wrap in single quotes and double internal quotes."""
    return "'" + p.replace("'", "''") + "'"


def discover_parquet_files(
    data_root: str,
    token_whitelist: Optional[Set[str]] = None
) -> List[str]:
    """
    Find all *.parquet under data_root. If token_whitelist is provided, only keep files
    whose parent folder is named like 'token_address=<address>' and the address is whitelisted.
    """
    root = pathlib.Path(data_root)
    if not root.exists():
        raise FileNotFoundError(f"{data_root} does not exist")

    all_parquets = [str(p) for p in root.rglob("*.parquet")]
    if token_whitelist is None:
        return all_parquets

    wl = {t.lower() for t in token_whitelist}
    keep: List[str] = []
    for f in all_parquets:
        # Walk up to find a 'token_address=<addr>' segment if present
        p = pathlib.Path(f)
        matched_addr = None
        for part in p.parents:
            name = part.name
            if name.startswith("token_address="):
                matched_addr = name.split("=", 1)[1].lower()
                break
        if matched_addr is None or matched_addr in wl:
            keep.append(f)
    return keep


def count_rows_in_all_parquet(
    data_root: str,
    token_whitelist: Optional[Set[str]] = None,
) -> int:
    """
    Count total rows across all parquet files found under data_root (optionally filtered
    by token_whitelist via directory name 'token_address=<address>').
    Returns the integer total row count.
    """
    files = discover_parquet_files(data_root, token_whitelist)
    if not files:
        raise FileNotFoundError(f"No .parquet files found under: {data_root}")

    con = duckdb.connect()
    try:
        con.execute(f"SET threads={CONFIG.duckdb_threads}")
        con.execute("SET enable_progress_bar=false")

        total_rows = 0
        n = len(files)
        print(f"Found {n:,} parquet files. Counting rows in chunks of {CONFIG.files_per_chunk}...")

        for i in range(0, n, CONFIG.files_per_chunk):
            chunk = files[i:i + CONFIG.files_per_chunk]
            paths_sql_list = ", ".join(_sql_quote_path(p) for p in chunk)

            # Count rows for this chunk
            # Note: read_parquet([...]) vertically concatenates the files.
            rows = con.execute(f"SELECT COUNT(*) FROM read_parquet([{paths_sql_list}])").fetchone()[0]
            total_rows += int(rows)

            # occasional progress
            processed = min(i + CONFIG.files_per_chunk, n)
            if (i // CONFIG.files_per_chunk + 1) % 10 == 0 or processed == n:
                print(f"  processed {processed:,} / {n:,} files — running total rows: {total_rows:,}")

        return total_rows
    finally:
        con.close()



DATA_ROOT = Location.PORTFOLIO_RECONSTRUCTION_DATA

total_rows = count_rows_in_all_parquet(DATA_ROOT)
half = total_rows / 2.0

print(f"\nTotal rows across all parquet files: {total_rows:,}")
print(f"Total rows / 2: {half:,.2f}")

In [ ]:
from __future__ import annotations

import os
import pathlib
import concurrent.futures
from dataclasses import dataclass
from typing import Dict, List, Optional, Set, Tuple

import duckdb
import pandas as pd  # <- needed for reading the token status CSV

from locations import Location


# --- Config ---
@dataclass
class Config:
    max_workers: int = min(32, os.cpu_count() or 1)
    duckdb_threads: int = min(8, os.cpu_count() or 1)
    files_per_chunk: int = 500  # how many parquet files per INSERT round

CONFIG = Config()


def _sql_quote_path(p: str) -> str:
    """DuckDB SQL string literal: wrap in single quotes and double internal single quotes."""
    return "'" + p.replace("'", "''") + "'"


# --- Whitelist loader ---
def load_token_whitelist_from_status_csv(csv_path: str) -> Set[str]:
    """
    Build a whitelist of contract addresses from a token status CSV.
    Keeps rows where:
      - is_erc_20 == TRUE
      - local_status == PASS
      - price_data_status == success
    Returns lowercased contract addresses.
    """
    if not os.path.exists(csv_path):
        print(f"Token status CSV not found: {csv_path}")
        return set()

    try:
        df = pd.read_csv(csv_path, dtype=str)

        def norm(s):
            return str(s).strip().lower()

        if "contract_address" not in df.columns:
            print("'contract_address' column missing in token status CSV")
            return set()

        mask = (
            df.get("is_erc_20", "").map(norm) == "true"
        ) & (
            df.get("local_status", "").map(norm) == "pass"
        ) & (
            df.get("price_data_status", "").map(norm) == "success"
        )

        addrs = (
            df.loc[mask, "contract_address"]
              .dropna()
              .astype(str)
              .str.strip()
              .str.lower()
              .tolist()
        )

        whitelist = {a for a in addrs if a.startswith("0x") and len(a) == 42}
        print(f"Token whitelist built: {len(whitelist)} ERC-20 tokens")
        return whitelist

    except Exception as e:
        print(f"Error reading token status CSV: {e}")
        return set()


# --- Discovery (parallel scan of token folders) ---
def discover_token_files_parallel(
    data_root: str,
    token_whitelist: Optional[Set[str]] = None
) -> Dict[str, List[str]]:
    """
    Discover token files in parallel across directory structure.

    Returns: {token_address: [file1.parquet, file2.parquet, ...]}
    Only tokens present in `token_whitelist` are returned (if provided).
    """
    def _scan_token_dir(token_dir: pathlib.Path) -> Tuple[Optional[str], List[str]]:
        if not token_dir.is_dir():
            return None, []
        name = token_dir.name
        if not name.startswith("token_address="):
            return None, []
        token = name.split("=", 1)[1].lower()

        # filter by whitelist early
        if token_whitelist and token not in token_whitelist:
            return None, []

        parquet_files = list(token_dir.rglob("*.parquet"))
        return token, [str(f) for f in parquet_files]

    base_path = pathlib.Path(data_root)
    token_dirs = list(base_path.glob("token_address=*"))
    if not token_dirs:
        raise FileNotFoundError(f"No token_address=* directories found under: {data_root}")

    result: Dict[str, List[str]] = {}
    with concurrent.futures.ThreadPoolExecutor(max_workers=CONFIG.max_workers) as executor:
        futures = [executor.submit(_scan_token_dir, td) for td in token_dirs]
        for future in concurrent.futures.as_completed(futures):
            token, files = future.result()
            if token and files:
                result[token] = files

    total_files = sum(len(v) for v in result.values())
    print(f"Discovered {len(result)} tokens with {total_files} files")
    return result


# --- Unique address extraction & write ---
def build_unique_addresses_parquet(
    data_root: str,
    output_parquet: str,
    token_whitelist: Optional[Set[str]] = None,
) -> int:
    """
    Reads all parquet files found under data_root/token_address=*/**/*.parquet
    (restricted to tokens in token_whitelist, if provided),
    extracts DISTINCT lower(address), and writes a single-column Parquet file
    with column 'address'. Returns the count of unique addresses.
    """
    discovered = discover_token_files_parallel(data_root, token_whitelist)
    all_files: List[str] = [p for files in discovered.values() for p in files]
    if not all_files:
        raise FileNotFoundError(
            "No parquet files found under the selected token set; "
            f"data_root={data_root}, whitelist_size={0 if token_whitelist is None else len(token_whitelist)}"
        )

    # Ensure output directory exists
    out_path = pathlib.Path(output_parquet)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    con = duckdb.connect()
    try:
        # Perf settings
        con.execute(f"SET threads={CONFIG.duckdb_threads}")
        con.execute("SET enable_progress_bar=false")

        # Temp staging table
        con.execute("CREATE TEMP TABLE addrs(address VARCHAR)")

        # Process in manageable chunks so the SQL isn't too large
        print("Collecting unique addresses...")
        for i in range(0, len(all_files), CONFIG.files_per_chunk):
            chunk = all_files[i:i + CONFIG.files_per_chunk]

            # Build a properly escaped list for DuckDB's read_parquet([...])
            paths_sql_list = ", ".join(_sql_quote_path(p) for p in chunk)

            con.execute(f"""
                INSERT INTO addrs
                SELECT DISTINCT lower(address) AS address
                FROM read_parquet([{paths_sql_list}])
            """)

            processed = min(i + CONFIG.files_per_chunk, len(all_files))
            if (i // CONFIG.files_per_chunk + 1) % 10 == 0 or processed == len(all_files):
                print(f"  processed {processed:,} / {len(all_files):,} files")

        # Final distinct + write to parquet
        print("Deduplicating and writing output...")
        con.execute("""
            CREATE TEMP VIEW unique_addrs AS
            SELECT DISTINCT address FROM addrs
            WHERE address IS NOT NULL AND length(address) > 0
        """)

        out_sql = _sql_quote_path(str(out_path))
        con.execute(f"""
            COPY (SELECT * FROM unique_addrs ORDER BY address)
            TO {out_sql}
            (FORMAT 'parquet')
        """)

        total_unique = con.execute("SELECT COUNT(*) FROM unique_addrs").fetchone()[0]
        print(f"✅ Wrote {total_unique:,} unique addresses to: {out_path}")
        return int(total_unique)

    finally:
        con.close()


DATA_ROOT = Location.PORTFOLIO_RECONSTRUCTION_DATA
TOKEN_STATUS_CSV = Location.TOKEN_STATUS_CSV
OUTPUT_PARQUET = str(Location.WHITELIST_ADDRESSES_PARQUET)

##for all TOKEN_STATUS_CSV = none

# Build whitelist and run
token_whitelist = load_token_whitelist_from_status_csv(TOKEN_STATUS_CSV)
total = build_unique_addresses_parquet(
        DATA_ROOT,
        OUTPUT_PARQUET,
        token_whitelist=token_whitelist if token_whitelist else None,
)
print("Total unique addresses:", f"{total:,}")
